# Pay for a point inside the street

Hard-margin SVM refuses every training point inside the street. One awkward point then forces the street to shrink, or makes the problem impossible.

Give each point a slack $\xi_i \ge 0$. A slack of $0$ means the point is outside or on the edge. A slack of $0.3$ means it missed the edge by $0.3$. The soft-margin problem is

$$
\begin{aligned}
\underset{\mathbf{w}, b, \boldsymbol{\xi}}{\operatorname{minimize}} \quad
& \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_i \xi_i \\
\text{subject to} \quad
& y_i(\mathbf{w}\cdot\mathbf{x}_i+b) \ge 1 - \xi_i, \\
& \xi_i \ge 0
\end{aligned}
$$

$C > 0$ is the price of one unit of slack. A large $C$ hates mistakes and narrow streets filled with slack. A small $C$ accepts slack in order to keep $\mathbf{w}$ short.


## The same cost, written as a hinge

At the best slacks, there is nothing to gain by making $\xi_i$ larger than the violation, and the constraint forbids making it smaller. So the best slack is exactly the hinge

$$
\xi_i = \max\bigl(0,\ 1 - y_i(\mathbf{w}\cdot\mathbf{x}_i+b)\bigr)
$$

The constrained problem is therefore the unconstrained problem

$$
\underset{\mathbf{w}, b}{\operatorname{minimize}} \quad
\frac{1}{2}\|\mathbf{w}\|^2
+ C\sum_i \max\bigl(0,\ 1 - y_i(\mathbf{w}\cdot\mathbf{x}_i+b)\bigr)
$$

Read the hinge on one point. If the functional margin is at least $1$, the hinge is $0$. If the margin is $0.5$, the hinge is $0.5$. If the point is on the wrong side and the margin is $-2$, the hinge is $3$. The hinge charges for entering the street, not only for crossing the center line.


## Two points, with $C = 1$

$$
x = 0,\ y = -1
\qquad
x = 1,\ y = +1
$$

The hard-margin solution puts the cut halfway and forces both margins to equal $1$:

$$
w = 2, \qquad b = -1, \qquad f(x) = 2x - 1
$$

Its hard objective is $\tfrac{1}{2}(2)^2 = 2$, with both slacks zero.

With $C = 1$ the dual multipliers cannot exceed $C$. Both hard-margin multipliers were $2$, so both clip to $1$. Stationarity then gives

$$
w = \alpha_2 \cdot (+1) \cdot 1 = 1
$$

The bias is not unique. For $w = 1$ the two hinges are

$$
\begin{aligned}
\xi(0) &= \max(0,\ 1 + b) \\
\xi(1) &= \max(0,\ -b)
\end{aligned}
$$

Every $b$ from $-1$ to $0$ has total slack $1$. Just outside that interval the total slack is larger. The balanced choice is the midpoint $b = -1/2$:

$$
f(0) = -\frac{1}{2}, \qquad f(1) = \frac{1}{2}
$$

Both points are on the correct side, both are inside the street, and each hinge equals $1/2$. The soft objective is

$$
\frac{1}{2}(1)^2 + 1\cdot\left(\frac{1}{2}+\frac{1}{2}\right) = \frac{3}{2}
$$

The hard solution, scored by this same soft objective, costs $2$. With $C = 1$ the shorter weight is the better trade.


In [1]:
# Hinge values for C = 1. Every bias from -1 to 0 costs the same.
from fractions import Fraction

def hinge(margin):
    return max(Fraction(0), 1 - margin)

def total_slack(b):
    # w = 1. Functional margins: y f(0) = -b, and y f(1) = 1 + b.
    return hinge(-b) + hinge(1 + b)

for b in (Fraction(-1), Fraction(-1, 2), Fraction(0)):
    assert total_slack(b) == 1
assert total_slack(Fraction(-11, 10)) > 1
assert total_slack(Fraction(1, 10)) > 1

soft_cost = Fraction(1, 2) * 1 + 1 * total_slack(Fraction(-1, 2))
hard_weight_cost = Fraction(1, 2) * 4 + 0
assert soft_cost == Fraction(3, 2)
assert hard_weight_cost == 2
print("soft cost", soft_cost)
print("hard weights scored by the soft cost", hard_weight_cost)


soft cost 3/2
hard weights scored by the soft cost 2


## Where the box $0 \le \alpha_i \le C$ comes from

Each slack has its own multiplier $\mu_i \ge 0$ for the constraint $\xi_i \ge 0$. The Lagrangian derivative in $\xi_i$ is

$$
C - \alpha_i - \mu_i = 0
$$

Hence $\alpha_i = C - \mu_i \le C$. Combined with $\alpha_i \ge 0$,

$$
0 \le \alpha_i \le C
$$

If $0 < \alpha_i < C$, then $\mu_i > 0$ and $\xi_i = 0$, so the point lies on the margin edge. If $\alpha_i = C$, the point is allowed to have positive slack. If $\alpha_i = 0$, the point is not pulling on $\mathbf{w}$.


## A pitfall

A positive slack is not the same thing as a wrong label. In the two-point example both predictions have the right sign, and both slacks are $1/2$. The street is wide because $\mathbf{w}$ is short, so the points fall inside it.

## What to carry forward

Soft-margin SVM is regularized hinge loss. $C$ is the price of slack. On the two-point line with $C = 1$, $w = 1$, every $b \in [-1, 0]$ is optimal, and the balanced choice $b = -1/2$ costs $3/2$.
